# GrowFields

**Compositional 4D Neural Fields for Topology-Changing Plant Growth**

[[Paper](https://link.springer.com/chapter/10.1007/978-3-032-37393-9_22)] [[arXiv](https://arxiv.org/abs/2607.03330)] [[Project page](https://joaquin-gajardo.github.io/growfields)] [[Code](https://github.com/Joaquin-Gajardo/growfields)] [[Data](https://hdl.handle.net/20.500.11850/806887)]

GrowFields models a growing plant as a set of *organs* (stem + leaves), each carrying
its own trajectory through time, driven by a single neural velocity field conditioned on
a learnable per-organ code.

This notebook is a tour of the package API: it loads one plant sequence, trains a
GrowFields model on it, and evaluates it. It trains a **short demo run on a single
plant**, not the paper configuration — the last section gives the commands that
reproduce the published tables.

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

## 1. Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import torch
    REPO_ROOT = "/content/growfields"
    !git clone https://github.com/Joaquin-Gajardo/growfields $REPO_ROOT

    # Install PyTorch3D
    !pip install --extra-index-url https://miropsota.github.io/torch_packages_builder pytorch3d==0.7.9+d9839a9pt{torch.__version__.replace("+", "")}

    # We just install plant-shape-analysis from requirements.txt, all the
    # rest should not be needed or already installed in Colab
    PSA_REQ = !grep '^plant-shape-analysis' {REPO_ROOT}/requirements.txt
    !pip install -q "{PSA_REQ[0]}"
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# The config files name `configs/` and `data/` by repo-relative path, so the
# working directory has to be the repo root.
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch

print(f"repo root: {REPO_ROOT}")
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — training will be slow. Runtime > Change runtime type > T4 GPU.")

## 2. Method

Equations below are transcribed from the paper's Method section; each is labelled with
the subsection it appears under (the paper does not label equations, so section names
are the stable reference) and paired with the code that implements it.

### 2.1 Organ-conditioned velocity field

*(Method § Formulation)* After canonical alignment, growth is a single continuous
velocity field shared by every organ. A SIREN $f_\theta$ maps a point, a time, and an
organ code to a velocity:

$$f_{\theta}: (\mathbf{x},\, t,\, \mathbf{z}_i) \;\mapsto\; \mathbf{v}(\mathbf{x}, t, \mathbf{z}_i) \in \mathbb{R}^{3}$$

where $\mathbf{x}$ is a 3D point in canonical coordinates, $t \in [0,1]$ is normalised
time, and $\mathbf{z}_i \in \mathbb{R}^{d}$ is a learnable latent code for organ $i$.
The code is concatenated with the spatio-temporal input, giving
$[\mathbf{x};\, t;\, \mathbf{z}_i] \in \mathbb{R}^{4+d}$. **The weights $\theta$ are
shared across all organs** — only $\mathbf{z}_i$ distinguishes them.

> **In code:** `SharedContinuousSiren` (`growfields/models/siren.py`). The config keys
> `embed_dim` ($d$) and `conditioning` select how $\mathbf{z}_i$ enters the network —
> `concat` for the equation above, `film` for FiLM modulation, or `none` to ablate
> conditioning entirely.

*(Method § Latent code design)* Following the auto-decoder paradigm there is no
encoder: the codes are optimised jointly with $\theta$, initialised from
$\mathcal{N}(\mathbf{0}, \sigma_z^2 \mathbf{I})$ and pulled toward the origin to keep
them bounded:

$$\mathcal{L}_{\mathrm{code}} = \frac{1}{N}\sum_{i=1}^{N} \lVert \mathbf{z}_i \rVert_2^2$$

> **In code:** weighted by `embed_reg_lambda`, with the codes on their own learning
> rate `lr_embed`.

### 2.2 Getting geometry from velocities

*(Method § Discrete dynamics)* The field predicts velocity, not position, so geometry
is obtained by integrating from the organ's first observed point cloud:

$$\mathbf{x}^{(t)} = \mathbf{x}^{(0)} + \sum_{\tau = 0,\, \Delta t,\, \dots,\, t-\Delta t} f_{\theta}\!\left(\mathbf{x}^{(\tau)}, \tau, \mathbf{z}_i\right) \Delta t$$

which is an Euler approximation of the continuous form
$\frac{d\mathbf{x}}{dt} = f_{\theta}(\mathbf{x}, t, \mathbf{z}_i)$.

> **In code:** `x_pred = x_pred + velocity * dt_sub` in `_integrate_segment`
> (`growfields/models/integration.py`). The subtlety is $\Delta t$: time is normalised
> over each organ's own **day** range, so `dt` comes from `make_t_norm_helpers` as
> `get_t_norm(t+1) - get_t_norm(t)` rather than being a constant `1/n_frames`. This is
> what makes the velocity *instantaneous* — per unit normalised time — so unevenly
> spaced scan days integrate correctly. Setting `use_odeint: true` swaps the Euler loop
> for a `torchdiffeq` solver.

### 2.3 Training objective

*(Method § Chamfer supervision)* Predictions are supervised at **every** timestep, not
just the last, by bidirectional Chamfer distance:

$$\mathcal{L}_{\mathrm{CD}}^{(i)} = \frac{1}{k-s}\sum_{t=s+1}^{k} \mathrm{CD}\!\left(\hat{\mathbf{X}}_i^{(t)},\, \mathbf{X}_i^{(t)}\right)$$

$$\mathrm{CD}(\hat{\mathbf{X}},\mathbf{X}) = \frac{1}{|\hat{\mathbf{X}}|}\sum_{\mathbf{x} \in \hat{\mathbf{X}}} \min_{\mathbf{y} \in \mathbf{X}} \lVert \mathbf{x}-\mathbf{y}\rVert_2^2 + \frac{1}{|\mathbf{X}|}\sum_{\mathbf{y} \in \mathbf{X}} \min_{\mathbf{x} \in \hat{\mathbf{X}}} \lVert \mathbf{x}-\mathbf{y}\rVert_2^2$$

*(Method § Total objective)* Averaged over the organs sampled in a mini-batch
$\mathcal{B}$ and added to the code regulariser:

$$\mathcal{L}_{\mathrm{total}} = \frac{1}{|\mathcal{B}|}\sum_{i \in \mathcal{B}} \mathcal{L}_{\mathrm{CD}}^{(i)} + \lambda_{\mathrm{code}} \mathcal{L}_{\mathrm{code}}$$

> **In code:** `growfields/models/siren_dynamics.py`, using `chamfer_distance` from
> pytorch3d. `organs_per_step` sets $|\mathcal{B}|$ and `embed_reg_lambda` is
> $\lambda_{\mathrm{code}}$.

## 3. Configuration

Experiments are driven by YAML configs under `configs/`, organised by the entry point
that consumes them:

| Directory | Consumed by | Model section |
|---|---|---|
| `configs/per_organ/` | `train_per_organ.py` | `default_model` + `per_organ_training` |
| `configs/full_plant/` | `train_fullplant.py` | `model` |

These are **not interchangeable** — a per-organ config trains one field per organ,
a full-plant config trains a single field for the whole plant.

`configs/per_organ/shared_continuous.yaml` is GrowFields as described above: one shared
organ-conditioned field.

In [ ]:
from omegaconf import OmegaConf

from growfields.pipeline.config import DEVICE, load_config, set_seed

CONFIG_PATH = "configs/per_organ/shared_continuous.yaml"

cfg = load_config(CONFIG_PATH)
print(OmegaConf.to_yaml(cfg))

Any key can be overridden. In the notebook, assign to it; from the command line, pass
the same dotted path as `key=value` (`python train_per_organ.py --config ... default_model.n_steps=5000`).

Below we shorten training drastically so the demo finishes in minutes. Everything else
is left at the paper setting.

In [ ]:
SEQUENCE = "maize_control_plant2"
DEMO_STEPS = 5000  # paper uses default_model.n_steps = 50000

cfg.dataset.sequence = SEQUENCE
cfg.default_model.n_steps = DEMO_STEPS
cfg.default_model.log_interval = 100

# Logging is opt-in and this notebook drives the model directly rather than through
# train_per_organ.py, so no Weights & Biases run is ever started.
cfg.wandb.enabled = False

set_seed(42)
print(f"sequence:   {cfg.dataset.sequence}")
print(f"model type: {cfg.default_model.model_type}")
print(f"n_steps:    {cfg.default_model.n_steps}  (demo; paper uses 50000)")
print(f"device:     {DEVICE}")

## 4. Data

[TrackPlant3D](https://www.sciencedirect.com/science/article/pii/S0168169924008263) gives
multi-day point cloud scans of maize, tomato, tobacco and sorghum with per-organ labels
and cross-day tracking. It downloads automatically on first use (229 MB) via the
`plant-shape-analysis` package.

Two things matter for per-organ training:

- **`normalize=False`.** Each organ is normalised independently by `OrganModelManager`,
  so global normalisation must be off.
- **The point split.** `configs/datasets/TrackPlant3D/point_splits/` ships the exact
  train/test point splits used in the paper. They are versioned here rather than
  regenerated, since the published numbers depend on them.

<details>
<summary><b>If the download fails on Colab</b> (HTTP 429 or 500)</summary>

The dataset host rate-limits anonymous downloads per IP, and Colab's addresses are
shared, so the automatic download sometimes fails there. Retrying the cell is worth a
shot first, since the failure is often transient. Otherwise it works from a browser, so
fetch the archive once and upload it:

1. Open <https://hdl.handle.net/20.500.11850/806887> and download `TrackPlant3D_v2.zip`.
2. In Colab's **Files** pane (folder icon, left sidebar), click **Upload to session
   storage** and pick the zip. It lands at `/content/TrackPlant3D_v2.zip`.
3. Run:

```python
!mkdir -p {REPO_ROOT}/data/TrackPlant3D/versions
!unzip -q "/content/TrackPlant3D_v2.zip" -d {REPO_ROOT}/data/TrackPlant3D/versions
```

`REPO_ROOT` comes from the setup cell above; the archive has to land in the repo's own
`data/` directory, which is where the config looks. That leaves
`{REPO_ROOT}/data/TrackPlant3D/versions/v2/`, so the next cell finds it and skips the
download.

Session storage is cleared when the runtime restarts, so this needs redoing after a
disconnect. Putting the zip in Google Drive and mounting it instead survives restarts.

</details>

In [ ]:
from growfields.pipeline.data import load_plant_dataset
from growfields.pipeline.splits import POINT_SPLITS_DIR

split_file = os.path.join(POINT_SPLITS_DIR, f"{SEQUENCE}_split.json")
assert os.path.exists(split_file), f"missing split file: {split_file}"

dataset = load_plant_dataset(
    cfg.dataset,
    SEQUENCE,
    split_file=split_file,
    timepoint_stride=1,       # every scan day
    normalize=False,          # per-organ normalisation happens in OrganModelManager
    device=DEVICE,
)

info = dataset.get_sequence_info()
print(f"{SEQUENCE}: {info['n_timepoints']} timepoints, "
      f"{info['n_organs']} organs ({info['n_leaves']} leaves + stem)")
print(f"  train/test points: {info['n_train']} / {info['n_test']}")

# One raw timepoint, to show what the pipeline is fed.
plant_ts = dataset.plant_dataset.get_timeseries_by_sequence_name(SEQUENCE)
tp = plant_ts["timepoints"][0]
print(f"\nkeys in a timepoint: {sorted(tp.keys())}")
print(f"  day:    {tp['day']}")
print(f"  points: {tp['points'].shape}")
print(f"  labels: organ ids {sorted(set(tp['labels'].tolist()))}  (0 = stem, >0 = leaves)")

## 5. Training

`OrganModelManager` is the per-organ entry point. It splits the plant into organ
sequences, normalises each one, and — for the shared model — trains a single field with
one latent code per organ, sampling `organs_per_step` organs per optimisation step.

Organs appear over time (a leaf that emerges on day 12 exists for fewer timepoints than
the stem), which the manager tracks per organ.

In [ ]:
from growfields.models.organ_model_manager import OrganModelManager

manager = OrganModelManager(
    dataset=dataset,
    default_config=OmegaConf.to_container(cfg.default_model, resolve=True),
    device=DEVICE,
)

organ_info = manager.get_organ_info()
print("Organs found:")
for organ_id, d in organ_info["organs"].items():
    print(f"  organ {organ_id:>2} ({d['type']:>7}): "
          f"present in {d['n_timepoints_present']}/{d['n_total_timepoints']} timepoints")

In [ ]:
manager.train_all_organs()

## 6. Evaluation

Training writes predicted point clouds; metrics are **recomputed from those files**
rather than cached during training, so there is one place the published numbers come
from. The layout below — `sequences/<name>/` under an experiment directory, plus an
`experiment_config.json` recording what produced it — is what the metrics scripts expect.

In [ ]:
from growfields.pipeline.results import prepare_results_dir, write_experiment_config

EXPERIMENT_DIR = "results/notebook_demo"
run_dir = os.path.join(EXPERIMENT_DIR, "sequences", SEQUENCE)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# Records the resolved config and which sequences ran. The third argument stands
# in for the command line a run was launched with; the readers of this file only
# ever look up `points_split_dir` in it, so that is all it needs to carry.
write_experiment_config(
    EXPERIMENT_DIR, cfg,
    {"points_split_dir": POINT_SPLITS_DIR},
    [SEQUENCE], config_file=CONFIG_PATH,
)
prepare_results_dir(run_dir, cfg, split_file)

manager.save_all_organs(run_dir, save_predictions=True, save_models=True,
                        save_metrics=True, save_full_plant=True)

The wrapper script computes full-plant metrics over the experiment directory and emits
the LaTeX row used in the paper's tables. With one sequence and a 1000-step model the
numbers will be well off the published ones — the point is the path from trained model
to table row.

In [ ]:
!PYTHONPATH=. bash scripts/compute_metrics_and_latex_table_fullplant.sh {EXPERIMENT_DIR} --no_std

## 7. Reproducing the paper tables

The published numbers come from the command-line entry points, not this notebook: they
run every test sequence at full step count, which is far longer than a Colab session.

```bash
# GrowFields (shared organ-conditioned field) on the test split
python train_per_organ.py \
    --config configs/per_organ/shared_continuous.yaml \
    --split test --n_runs 5 \
    --points_split_dir configs/datasets/TrackPlant3D/point_splits

# Table 1 (full-plant) and Table 2 (per-organ) from a finished experiment directory
bash scripts/compute_metrics_and_latex_table_fullplant.sh <experiment_dir>
bash scripts/compute_metrics_and_latex_table_per_organ.sh <experiment_dir>
```

Swap the config to compare method variants:

| Config | Method |
|---|---|
| `configs/per_organ/shared_continuous.yaml` | GrowFields — one shared organ-conditioned field |
| `configs/per_organ/continuous.yaml` | One independent continuous field per organ |
| `configs/per_organ/pairwise.yaml` | DPF per-organ (pairwise + fine-tuning) |
| `configs/per_organ/encoder_decoder.yaml` | COAP baseline — encoder–decoder with a per-organ latent |
| `configs/full_plant/pairwise.yaml` | DPF single-model, via `train_fullplant.py` |

Ablations are config overrides on top of these, e.g. `default_model.conditioning=none`
to remove organ conditioning, `dataset.aligned_leaves=false` to drop canonicalisation, or
`default_model.use_multistep_loss=false` to supervise only the final frame.

`REPRODUCIBILITY.md` carries every published table with its run-to-run standard
deviations and the exact command behind each one, including the run counts, which differ
by table. See the [README](https://github.com/Joaquin-Gajardo/growfields) for the full set
of flags, and `configs/README.md` for the config schema.